# Data Creation Playground

Full multi-depth pruning pipeline. G = Gemma-4 (Colab GPU), D = Gemini Flash Lite (cloud).
Source = `avreymi/reasoning-spectrum-qa` (1000 diverse QA across 6 reasoning families); each
question is assembled with its context and choices via `format_spectrum_question`, and answer
fields are never shown to G.

**Before running:** Enable GPU runtime → Runtime → Change runtime type → T4 GPU (or A100).

In [1]:
!git clone https://github.com/avrymi-asraf/reasoning-pruning.git
%cd reasoning-pruning
# Gemma 4 requires transformers from git main — not yet in a stable PyPI release
%pip install -q "git+https://github.com/huggingface/transformers.git" "accelerate>=0.34.0" "datasets>=4.8.5" "torchvision>=0.27.0" "pyyaml>=6.0.2"

fatal: destination path 'reasoning-pruning' already exists and is not an empty directory.
/content/reasoning-pruning
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!git pull

Already up to date.


In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [3]:
import os
os.environ["HF_TOKEN"] = input("Enter your Hugging Face token: ")
os.environ["GEMINI_API_KEY"] = input("Enter your Gemini API key: ")

In [5]:
import os
import sys
sys.path.insert(0, "src")

from pathlib import Path
from dataclasses import replace

from reasoning_pruning.data_creation import (
    load_data_creation_config,
    load_questions,
    build_pt_dataset,
    build_rows_for_question,
    format_context,
    advance_context_units,
    build_pruning_transition_row,
    split_reasoning_units
)
from reasoning_pruning.clients import (
    TransformersGenerator,
    create_decision_model_from_config,
    load_prompt_template,
)
from reasoning_pruning.pipeline_inspection import run_pipeline_inspection

print("Imports OK")

Imports OK


In [6]:
# Downloads ~5GB from Hub — takes 1-2 min on first run
generator = TransformersGenerator(
    source_model="avreymi/gemma-4-E2B-it-reasoning-pruning",
    generation_config={"max_new_tokens": 100, "temperature": 0.7, "do_sample": True},
    max_units_per_batch=2,
)
print("G ready:", generator.source_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

G ready: avreymi/gemma-4-E2B-it-reasoning-pruning


In [13]:
# Load config from YAML, then override depth/limit for quick playground runs.
# Source is avreymi/reasoning-spectrum-qa — questions are pulled straight from it,
# with context + choices already assembled by format_spectrum_question.
config = load_data_creation_config(Path("configs/data/dataset_builder_spectrum_gemma4.yaml"))
config = replace(config, max_pruning_depth=4, max_examples_per_question=2, source_limit=8, max_units_per_batch=20)

questions = load_questions(config, hf_token=os.environ.get("HF_TOKEN"))

print(f"Config: max_depth={config.max_pruning_depth}, G={config.generator['model_id']}")
print(f"Source: {config.source_dataset} (split={config.source_split})")
print(f"Questions: {len(questions)}")
print("\n--- Example assembled question (context + choices, no answer) ---\n")
print(questions[0])

Config: max_depth=4, G=avreymi/gemma-4-E2B-it-reasoning-pruning
Source: avreymi/reasoning-spectrum-qa (split=data)
Questions: 8

--- Example assembled question (context + choices, no answer) ---

The hypothalamus also produces hormones that directly regulate body processes. For example, it produces antidiuretic hormone. This hormone travels to the kidneys and stimulates them to conserve water by producing more concentrated urine.

What produces hormones that directly regulate body processes?

Options:
(A) pancreas
(B) lymph glands
(C) hypothalamus
(D) hippocampus


## D-prompt workflow

Iterate the decision-model prompt without leaving the notebook — no extra dependencies. Three cells:

1. **List / view** — see every `prompts/*.txt` and print any one inline with `show_prompt("<stem>")`.
2. **Write / edit** — set `PROMPT_NAME` + `PROMPT_TEXT` to create a new version (or overwrite an existing one with `OVERWRITE = True`). Only `{prompt_version} {question} {context} {reasoning_units}` may appear as single braces; escape any others as `{{ }}`.
3. **Choose** — set `PROMPT_VERSION` to any stem; it overrides `config.decision["prompt_version"]` and rebuilds D from the config (the production path). Re-run the build cells below to compare removal rates across versions.

> **D model note:** D is now built from `config.decision`, so it uses the model in `configs/data/dataset_builder_spectrum_gemma4.yaml` — currently `gemini-3.1-flash-lite` (matches `CLAUDE.md`). This replaces the notebook's old hardcoded `gemini-flash-lite-latest`. If `gemini-3.1-flash-lite` is not valid for your key, change `model_id` in that YAML.

In [8]:
# LIST / VIEW prompts. PROMPTS_DIR resolves against the kernel's working dir —
# the printed absolute path must point at the repo's prompts/ folder. If it does
# not (e.g. VSCode started the kernel in the notebook's own dir), the import and
# config cells above would already have failed; fix the kernel cwd to the repo root.
PROMPTS_DIR = "prompts"


def list_prompts() -> list[str]:
    return sorted(p.stem for p in Path(PROMPTS_DIR).glob("*.txt"))


def show_prompt(version: str) -> None:
    print(load_prompt_template(version, PROMPTS_DIR))


print("prompts/ resolves to:", Path(PROMPTS_DIR).resolve())
print("\nAvailable prompt versions:")
for name in list_prompts():
    print("  -", name)

# Read one inline before choosing/editing:
# show_prompt("conservative-skip-v2-general")

prompts/ resolves to: /content/reasoning-pruning/prompts

Available prompt versions:
  - balanced-skip-v1
  - conservative-skip-v1
  - conservative-skip-v2-family-aware
  - conservative-skip-v2-general
  - incremental-skip-v2


In [9]:
# CHOOSE the prompt D uses. Set PROMPT_VERSION to any stem from the list above.
# This overrides config.decision["prompt_version"] and rebuilds D from the config
# — the same path the CLI and HF Jobs use, so D runs at config.pruning's
# temperature: 0.0 for fair prompt-to-prompt comparison.
PROMPT_VERSION = "conservative-skip-v1"

config.decision["prompt_version"] = PROMPT_VERSION
decision_model = create_decision_model_from_config(
    config.decision, config.pruning, prompts_dir=PROMPTS_DIR
)

print("D ready:", config.decision["model_id"], "| prompt:", config.decision["prompt_version"])
print("-" * 70)
show_prompt(PROMPT_VERSION)

D ready: gemini-3.1-flash-lite | prompt: conservative-skip-v1
----------------------------------------------------------------------
Decision prompt version: {prompt_version}

You are a conservative pruning decision model. Your job: find the first reasoning unit that is pure filler and can be removed without any loss of correctness.

STRICT REMOVAL CONDITIONS — all must hold:
1. The unit contains NO computation, NO numeric value, NO logical deduction, and NO new fact. It is pure filler (e.g. a numbering artifact, a commentary, a restatement of the problem, or a statement of intent).
2. The unit at index removed_end_index+1 — which becomes the training target — contains ACTUAL reasoning: a numeric computation, a derived fact, or a logical deduction. It must NOT be another goal/intent statement ('Determine X', 'We need to find Y', 'Calculate Z', 'Convert A to B').
3. Removing the span leaves the reasoning coherent.

REMOVABLE examples: '1.' (bare numbering), 'Let me think.' (filler), 'Th

In [10]:
def show_rows(rows: list[dict]) -> None:
    if not rows:
        print("No rows generated — D found no safe removals at any depth")
        return
    for row in rows:
        sep = "=" * 70
        print(f"\n{sep}")
        print(f"  Depth {row['pruning_depth']}")
        print(f"{sep}")
        print(f"\n[GENERATED UNITS]")
        for i, u in enumerate(row["generated_units"]):
            marker = "  ✗" if row["metadata"]["removed_start_index"] <= i <= row["metadata"]["removed_end_index"] else "   "
            print(f"{marker} {i}: {u}")
        print(f"\n[REMOVED] indices {row['metadata']['removed_start_index']}–{row['metadata']['removed_end_index']}")
        print(f"  Reason: {row['metadata']['decision_reason']}")
        print(f"\n[INPUT_X]")
        for line in row["input_x"].splitlines():
            print(f"  {line}")
        print(f"\n[TARGET_Y]")
        print(f"  {row['target_y']}")
    print(f"\n  → {len(rows)} training row(s) from this question")

In [15]:
# Run on a single question to inspect each depth in detail
question_index = 2
print(f"Inspecting question {question_index}:\n{questions[question_index]}\n")
rows = build_rows_for_question(
    question=questions[question_index],
    generator=generator,
    decision_model=decision_model,
    config=config,
)
show_rows(rows)

Inspecting question 2:
The half-hour newscast includes 12 minutes of national news, 5 minutes of international news, 5 minutes of sports, and 2 minutes of weather forecasts. The rest is advertisements. How many minutes of advertising are in the newscast?

No rows generated — D found no safe removals at any depth


In [11]:
import pprint


pprint.pprint(config)

DataCreationConfig(round_id='spectrum-gemma4-r2',
                   source_type='hf_dataset',
                   source_dataset='avreymi/reasoning-spectrum-qa',
                   source_dataset_revision='main',
                   source_questions_path=None,
                   source_subset=None,
                   source_split='data',
                   source_question_field='question',
                   source_limit=8,
                   code_version='local-dev',
                   hub_dataset_id='avreymi/reasoning-pruning-pt-spectrum-gemma4-r2',
                   private=True,
                   generator={'model_id': 'avreymi/gemma-4-E2B-it-reasoning-pruning',
                              'provider': 'transformers'},
                   decision={'api_key_env': 'GEMINI_API_KEY',
                             'model_id': 'gemini-3.1-flash-lite',
                             'prompt_version': 'conservative-skip-v1',
                             'provider': 'gemini-json'},
         

In [15]:
# Run the same pipeline inspection path as scripts/pipeline_inspection.py.
# Change QUESTION_INDEX to inspect a different source question.
QUESTION_INDEX = 0
question = questions[QUESTION_INDEX]
rows = run_pipeline_inspection(
    question=question,
    generator=generator,
    decision_model=decision_model,
    config=config,
)



Original question
The hypothalamus also produces hormones that directly regulate body processes. For example, it produces antidiuretic hormone. This hormone travels to the kidneys and stimulates them to conserve water by producing more concentrated urine.

What produces hormones that directly regulate body processes?

Options:
(A) pancreas
(B) lymph glands
(C) hypothalamus
(D) hippocampus

Depth 0

[Context before generation]
Question:
The hypothalamus also produces hormones that directly regulate body processes. For example, it produces antidiuretic hormone. This hormone travels to the kidneys and stimulates them to conserve water by producing more concentrated urine.

What produces hormones that directly regulate body processes?

Options:
(A) pancreas
(B) lymph glands
(C) hypothalamus
(D) hippocampus

----------------------------------------------------------------------
Depth 0 / attempt 1
----------------------------------------------------------------------

[G generated reasonin